# K-MIMIC-MEDS — Dataset Validation

This notebook checks that the dataset produced by the ETL pipeline conforms to the MEDS format.

**Pipeline :**
1. `pre_meds.py` — transforms raw `.xlsx` files into intermediate `.parquet` files
2. `meds_convert.py` — converts intermediate `.parquet` files into final MEDS dataset

**Expected Structure :**
```
data/output/
├── data/
│   ├── train/0.parquet
│   ├── tuning/0.parquet
│   └── held_out/0.parquet
└── metadata/
    ├── codes.parquet
    ├── dataset.json
    └── subject_splits.parquet
```

In [1]:
import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("data/output")

## 1. File Structure

In [2]:
# Check that all expected files exist
expected = [
    "data/train/0.parquet",
    "data/tuning/0.parquet",
    "data/held_out/0.parquet",
    "metadata/codes.parquet",
    "metadata/dataset.json",
    "metadata/subject_splits.parquet",
]

for f in expected:
    path = OUTPUT_DIR / f
    status = "✅" if path.exists() else "❌ MISSING"
    print(f"{status}  {f}")

✅  data/train/0.parquet
✅  data/tuning/0.parquet
✅  data/held_out/0.parquet
✅  metadata/codes.parquet
✅  metadata/dataset.json
✅  metadata/subject_splits.parquet


## 2. MEDS Schema

In [3]:
# Load the train split
train = pd.read_parquet(OUTPUT_DIR / "data/train/0.parquet")

print("Column types :")
print(train.dtypes)
print()

# Compliance checks
checks = {
    "subject_id is int64": str(train["subject_id"].dtype) in ("int64", "Int64"),
    "time is datetime": "datetime" in str(train["time"].dtype),
    "code is string/object": str(train["code"].dtype) in ("object", "string", "str"),
    "numeric_value is float32": str(train["numeric_value"].dtype) == "float32",
    "exactly 4 columns": len(train.columns) == 4,
}

for check, result in checks.items():
    print(f"{'✅' if result else '❌'}  {check}")

Column types :
subject_id                Int64
time             datetime64[us]
code                        str
numeric_value           float32
dtype: object

✅  subject_id is int64
✅  time is datetime
✅  code is string/object
✅  numeric_value is float32
✅  exactly 4 columns


## 3. Preview of a patient's data

In [4]:
# Take the first patient and display their complete history
first_patient = train["subject_id"].iloc[0]
patient_df = train[train["subject_id"] == first_patient]

print(f"Patient {first_patient} — {len(patient_df)} events")
patient_df.head(20)

Patient 4056806253170987 — 263 events


,subject_id,time,code,numeric_value
0,4056806253170987,NaT,GENDER//M,NaN
1,4056806253170987,2090-01-01 00:00:01,MEDS_BIRTH,NaN
2,4056806253170987,2130-10-29 11:32:28,ED_REGISTRATION,NaN
3,4056806253170987,2130-10-29 11:53:28,ED_OUT,NaN
4,4056806253170987,2130-10-29 12:38:28,HOSPITAL_ADMISSION//Emergency department,NaN
5,4056806253170987,2130-10-29 12:38:28,INSURANCE//Others,NaN
6,4056806253170987,2130-10-29 12:38:28,MARITAL_STATUS//single,NaN
7,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//I251,NaN
8,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//I839,NaN
9,4056806253170987,2130-10-29 12:38:28,DIAGNOSIS//KCD8//Others,NaN


## 4. Static events (time = NaT)

In [5]:
# Static events should be at the top of each patient's record
static = train[train["time"].isna()]
print(f"Number of static events : {len(static)}")
print()
print("Types of static events :")
print(static["code"].apply(lambda x: x.split("//")[0]).value_counts())

Number of static events : 1062

Types of static events :
code
GENDER    1062
Name: count, dtype: int64


## 5. Codes — Quality

In [7]:
# Check absence of //nan in codes
nan_codes = train[train["code"].str.contains("//nan", na=False)]
print(f"Codes containing //nan : {len(nan_codes)} {'✅' if len(nan_codes) == 0 else '❌'}")

# Check absence of UNKNOWN codes
unknown_codes = train[train["code"] == "UNKNOWN"]
print(f"UNKNOWN codes : {len(unknown_codes)}")

print()
print("Present code prefixes :")
present_prefixes = train["code"].dropna().astype(str).apply(lambda x: x.split("//")[0])
print(present_prefixes.value_counts())

Codes containing //nan : 0 ✅
UNKNOWN codes : 0

Present code prefixes :
code
CHARTEVENT            285704
LAB                   102151
OUTPUT                 28861
INPUT_START            21242
MEDICATION              4491
DIAGNOSIS               2466
PROCEDURE_ICD           1175
GENDER                  1062
HOSPITAL_ADMISSION      1062
MARITAL_STATUS          1062
ICU_ADMISSION           1062
ICU_DISCHARGE           1062
HOSPITAL_DISCHARGE      1062
INSURANCE               1061
MEDS_BIRTH              1001
ED_REGISTRATION          461
ED_OUT                   461
MEDS_DEATH               144
Name: count, dtype: int64


In [8]:
# Most frequent codes
print("Top 15 most frequent codes :")
train["code"].value_counts().head(15)

Top 15 most frequent codes :


code
CHARTEVENT//001C_1021_25105//회/min    26787
CHARTEVENT//001C_1023_27310//회/min    26786
CHARTEVENT//001C_1013_25640//mmHg     25111
CHARTEVENT//001C_1014_27095//mmHg     25085
CHARTEVENT//001C_1012_24795//mmHg     25050
CHARTEVENT//001C_1016_25640//mmHg     21842
CHARTEVENT//001C_1015_24795//mmHg     21799
CHARTEVENT//001C_1017_27095//mmHg     21792
OUTPUT//001O_1481_25470//cc           21721
CHARTEVENT//001C_1026_26520//℃        21048
INPUT_START//001I_1315_26175//cc      17253
CHARTEVENT//001C_1003_24480//%        16045
CHARTEVENT//001C_1894_20720//%         7446
CHARTEVENT//001C_1710_22255//L/min     7238
CHARTEVENT//001C_1247_27285//ml        7233
Name: count, dtype: int64

## 6. Split Distribution

In [9]:
splits = pd.read_parquet(OUTPUT_DIR / "metadata/subject_splits.parquet")

counts = splits["split"].value_counts()
total = len(splits)

print(f"Total patients : {total}")
print()
for split_name, count in counts.items():
    pct = count / total * 100
    print(f"  {split_name:10} : {count:5} patients ({pct:.1f}%)")

# Check proportions
print()
checks = {
    "train ~80%": 75 <= counts.get("train", 0) / total * 100 <= 85,
    "tuning ~10%": 8 <= counts.get("tuning", 0) / total * 100 <= 12,
    "held_out ~10%": 8 <= counts.get("held_out", 0) / total * 100 <= 12,
}
for check, result in checks.items():
    print(f"{'✅' if result else '❌'}  {check}")

Total patients : 1328

  train      :  1062 patients (80.0%)
  held_out   :   134 patients (10.1%)
  tuning     :   132 patients (9.9%)

✅  train ~80%
✅  tuning ~10%
✅  held_out ~10%


## 7. Metadata — codes.parquet

In [10]:
codes = pd.read_parquet(OUTPUT_DIR / "metadata/codes.parquet")

print(f"Unique codes : {len(codes)}")
print(f"Codes with description : {codes['description'].notna().sum()}")
print(f"Codes without description : {codes['description'].isna().sum()}")
print()
print("Examples of codes with description :")
codes[codes["description"].notna()][["code", "description"]].head(10)

Unique codes : 174
Codes with description : 83
Codes without description : 91

Examples of codes with description :


,code,description
0,CHARTEVENT//001C_1003_24480//%,SpO2 > 산소포화도
1,CHARTEVENT//001C_1012_24795//mmHg,SBP > 수축기혈압
2,CHARTEVENT//001C_1013_25640//mmHg,DBP > 이완기혈압
3,CHARTEVENT//001C_1014_27095//mmHg,mean BP(연동) > 평균혈압(연동)
4,CHARTEVENT//001C_1015_24795//mmHg,ASBP > 수축기혈압
5,CHARTEVENT//001C_1016_25640//mmHg,ADBP > 이완기혈압
6,CHARTEVENT//001C_1017_27095//mmHg,mean ABP > 평균혈압(연동)
7,CHARTEVENT//001C_1021_25105//회/min,HR > 심박동수
8,CHARTEVENT//001C_1023_27310//회/min,RR > 호흡수
9,CHARTEVENT//001C_1026_26520//℃,BT > 체온


## 8. Metadata — dataset.json

In [11]:
meta = json.loads((OUTPUT_DIR / "metadata/dataset.json").read_text())
for key, value in meta.items():
    print(f"  {key:20} : {value}")

  dataset_name         : K-MIMIC-MEDS
  dataset_version      : 0.2.0
  etl_name             : kmimic-meds
  etl_version          : 0.2.0
  meds_version         : 0.3.3
  created_at           : 2026-04-14T07:59:43.480692+00:00


## 9. Global Statistics

In [12]:
# Load all splits
all_dfs = []
for split_name in ["train", "tuning", "held_out"]:
    df = pd.read_parquet(OUTPUT_DIR / f"data/{split_name}/0.parquet")
    df["split"] = split_name
    all_dfs.append(df)

full = pd.concat(all_dfs, ignore_index=True)

print("=== Global Statistics ===")
print(f"Total events         : {len(full):,}")
print(f"Total patients       : {full['subject_id'].nunique():,}")
print(f"Static events        : {full['time'].isna().sum():,}")
print(f"Dynamic events       : {full['time'].notna().sum():,}")
print(f"With numeric value   : {full['numeric_value'].notna().sum():,}")
print()
print("Events per split :")
print(full.groupby("split").agg(
    events=("code", "count"),
    patients=("subject_id", "nunique")
))

=== Global Statistics ===
Total events         : 639,454
Total patients       : 1,328
Static events        : 1,328
Dynamic events       : 638,126
With numeric value   : 605,941

Events per split :
          events  patients
split                     
held_out   61849       134
train     455590      1062
tuning     55958       132


## 10. Validation Summary

In [13]:
print("=== VALIDATION SUMMARY ===")
print()

all_checks = {
    "Files present"              : all((OUTPUT_DIR / f).exists() for f in expected),
    "subject_id is int64"        : str(train["subject_id"].dtype) in ("int64", "Int64"),
    "time is datetime"           : "datetime" in str(train["time"].dtype),
    "numeric_value is float32"   : str(train["numeric_value"].dtype) == "float32",
    "No //nan in codes"          : len(train[train["code"].str.contains("//nan", na=False)]) == 0,
    "Static events in NaT"       : train[train["time"].isna()]["code"].str.startswith(("GENDER", "DIAGNOSIS")).all(),
    "Splits 80/10/10"            : all(checks.values()),
    "codes.parquet enriched"     : codes["description"].notna().sum() > 0,
    "dataset.json present"       : (OUTPUT_DIR / "metadata/dataset.json").exists(),
}

for check, result in all_checks.items():
    print(f"{'✅' if result else '❌'}  {check}")

print()
passed = sum(all_checks.values())
total_checks = len(all_checks)
print(f"Result: {passed}/{total_checks} checks passed")

=== VALIDATION SUMMARY ===

✅  Files present
✅  subject_id is int64
✅  time is datetime
✅  numeric_value is float32
✅  No //nan in codes
✅  Static events in NaT
✅  Splits 80/10/10
✅  codes.parquet enriched
✅  dataset.json present

Result: 9/9 checks passed


In [14]:
# Check that diagnoses_icd has real timestamps
diag = train[train["code"].str.startswith("DIAGNOSIS")]
print(f"Total diagnoses: {len(diag)}")
print(f"With timestamp   : {diag['time'].notna().sum()}")
print(f"Without timestamp: {diag['time'].isna().sum()}")
print()
print(diag[["subject_id", "time", "code"]].head(10))

Total diagnoses: 2466
With timestamp   : 2466
Without timestamp: 0

             subject_id                time                     code
7      4056806253170987 2130-10-29 12:38:28    DIAGNOSIS//KCD8//I251
8      4056806253170987 2130-10-29 12:38:28    DIAGNOSIS//KCD8//I839
9      4056806253170987 2130-10-29 12:38:28  DIAGNOSIS//KCD8//Others
268   14397825242364671 2846-06-20 02:56:00  DIAGNOSIS//KCD8//Others
789   21501110057397942 2387-11-01 00:00:18    DIAGNOSIS//KCD8//I509
790   21501110057397942 2387-11-01 00:00:18  DIAGNOSIS//KCD8//Others
1115  42028212112143695 2319-08-09 20:49:29    DIAGNOSIS//KCD8//N179
1116  42028212112143695 2319-08-09 20:49:29  DIAGNOSIS//KCD8//Others
1117  42028212112143695 2319-08-09 20:49:29    DIAGNOSIS//KCD8//Z944
1814  47909883778226412 2679-12-15 22:28:35  DIAGNOSIS//KCD8//Others
